In [1]:
from pyspark.sql import SparkSession, functions as F, types as T

In [4]:
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("spark-hw")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/18 09:32:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
print(spark.version)

3.5.5


In [10]:
actor_df = spark.read.csv('/opt/workspace/data/actor.csv', header=True, inferSchema=True)
address_df = spark.read.csv('/opt/workspace/data/address.csv', header=True, inferSchema=True)
category_df = spark.read.csv('/opt/workspace/data/category.csv', header=True, inferSchema=True)
city_df = spark.read.csv('/opt/workspace/data/city.csv', header=True, inferSchema=True)
country_df = spark.read.csv('/opt/workspace/data/country.csv', header=True, inferSchema=True)
customer_df = spark.read.csv('/opt/workspace/data/customer.csv', header=True, inferSchema=True)
film_df = spark.read.csv('/opt/workspace/data/film.csv', header=True, inferSchema=True)
film_actor_df = spark.read.csv('/opt/workspace/data/film_actor.csv', header=True, inferSchema=True)
film_category_df = spark.read.csv('/opt/workspace/data/film_category.csv', header=True, inferSchema=True)
inventory_df = spark.read.csv('/opt/workspace/data/inventory.csv', header=True, inferSchema=True)
language_df = spark.read.csv('/opt/workspace/data/language.csv', header=True, inferSchema=True)
payment_df = spark.read.csv('/opt/workspace/data/payment.csv', header=True, inferSchema=True)
rental_df = spark.read.csv('/opt/workspace/data/rental.csv', header=True, inferSchema=True)
staff_df = spark.read.csv('/opt/workspace/data/staff.csv', header=True, inferSchema=True)
store_df = spark.read.csv('/opt/workspace/data/store.csv', header=True, inferSchema=True)

In [12]:
film_df.show()


+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+--------------------+--------------------+
|film_id|           title|         description|release_year|language_id|original_language_id|rental_duration|rental_rate|length|replacement_cost|rating|         last_update|    special_features|            fulltext|
+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+--------------------+--------------------+
|      1|ACADEMY DINOSAUR|A Epic Drama of a...|        2006|          1|                NULL|              6|       0.99|    86|           20.99|    PG|2022-09-10 16:46:...|{Deleted Scenes,B...|'academi':1 'batt...|
|      2|  ACE GOLDFINGER|A Astounding Epis...|        2006|          1|                NULL|              3|       4.99|    48|        

In [16]:
films_by_category_count_df = (category_df
                             .join(film_category_df, on="category_id")
                             .groupBy("name")
                             .agg(F.count("film_id").alias("film_count"))
                             .orderBy(F.desc("film_count")))

In [25]:
films_by_category_count_df.show(truncate=False)

+-----------+----------+
|name       |film_count|
+-----------+----------+
|Sports     |74        |
|Foreign    |73        |
|Family     |69        |
|Documentary|68        |
|Animation  |66        |
|Action     |64        |
|New        |63        |
|Drama      |62        |
|Games      |61        |
|Sci-Fi     |61        |
|Children   |60        |
|Comedy     |58        |
|Travel     |57        |
|Classics   |57        |
|Horror     |56        |
|Music      |51        |
+-----------+----------+



In [18]:
top_actors_df = (
    actor_df
    .join(film_actor_df, "actor_id")
    .join(film_df, "film_id")
    .join(inventory_df, "film_id")
    .join(rental_df, "inventory_id")
    .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
    .groupBy("actor_id", "full_name")
    .agg(F.count("*").alias("rental_count"))
    .orderBy(F.desc("rental_count"))
    .limit(10)
)

In [19]:
top_actors_df.show()

[Stage 39:>                                                         (0 + 1) / 1]

+--------+------------------+------------+
|actor_id|         full_name|rental_count|
+--------+------------------+------------+
|     107|    GINA DEGENERES|         753|
|     181|    MATTHEW CARREY|         678|
|     198|       MARY KEITEL|         674|
|     144|ANGELA WITHERSPOON|         654|
|     102|       WALTER TORN|         640|
|      60|       HENRY BERRY|         612|
|     150|       JAYNE NOLTE|         611|
|      37|        VAL BOLGER|         605|
|      23|     SANDRA KILMER|         604|
|      90|      SEAN GUINESS|         599|
+--------+------------------+------------+



In [20]:
top_category_df = (
    category_df
        .join(film_category_df, "category_id")
        .join(inventory_df, "film_id")
        .join(rental_df, "inventory_id")
        .join(payment_df, "rental_id")
        .groupBy("category_id", "name")
        .agg(F.sum("amount").alias("total_spent"))
        .orderBy(F.desc("total_spent"), F.asc("name"))
        .limit(1)
)

In [21]:
top_category_df.show()

[Stage 46:>                                                         (0 + 1) / 1]

+-----------+------+-----------------+
|category_id|  name|      total_spent|
+-----------+------+-----------------+
|         15|Sports|5314.209999999843|
+-----------+------+-----------------+



In [22]:
films_not_in_inventory_df = (
    film_df
        .join(inventory_df, on="film_id", how="left_anti")
        .select("title")
)

In [24]:
films_not_in_inventory_df.show(truncate=False)

+----------------------+
|title                 |
+----------------------+
|ALICE FANTASIA        |
|APOLLO TEEN           |
|ARGONAUTS TOWN        |
|ARK RIDGEMONT         |
|ARSENIC INDEPENDENCE  |
|BOONDOCK BALLROOM     |
|BUTCH PANTHER         |
|CATCH AMISTAD         |
|CHINATOWN GLADIATOR   |
|CHOCOLATE DUCK        |
|COMMANDMENTS EXPRESS  |
|CROSSING DIVORCE      |
|CROWDS TELEMARK       |
|CRYSTAL BREAKING      |
|DAZED PUNK            |
|DELIVERANCE MULHOLLAND|
|FIREHOUSE VIETNAM     |
|FLOATS GARDEN         |
|FRANKENSTEIN STRANGER |
|GLADIATOR WESTWARD    |
+----------------------+
only showing top 20 rows



In [28]:
children_top3_actors_df = (
    category_df
    .filter(F.col("name") == "Children")
    .join(film_category_df, on="category_id")          
    .join(film_actor_df, on="film_id")                 
    .join(actor_df, on="actor_id")                     
    .withColumn("full_name", F.concat_ws(" ", "first_name", "last_name"))
    .groupBy("actor_id", "full_name")
    .agg(F.count("*").alias("appearances"))
    .orderBy(F.desc("appearances"), F.asc("full_name"))
    .limit(3)
)

In [29]:
children_top3_actors_df.show(truncate=False)

[Stage 64:>                                                         (0 + 1) / 1]

+--------+-------------+-----------+
|actor_id|full_name    |appearances|
+--------+-------------+-----------+
|17      |HELEN VOIGHT |7          |
|127     |KEVIN GARLAND|5          |
|66      |MARY TANDY   |5          |
+--------+-------------+-----------+



In [30]:
spark.stop()